## 1. Завантаження всіх результатів


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.pipelines.evaluation_pipeline import compare_horizons, compare_window_types

processed_dir = Path('data/processed')
results_a_h1 = pd.read_csv(processed_dir / 'results_experiment_a_h1.csv', index_col=0)
results_a_h7 = pd.read_csv(processed_dir / 'results_experiment_a_h7.csv', index_col=0)
results_b_h1 = pd.read_csv(processed_dir / 'results_experiment_b_h1.csv', index_col=0)
results_b_h7 = pd.read_csv(processed_dir / 'results_experiment_b_h7.csv', index_col=0)
regimes = pd.read_csv(processed_dir / 'results_regimes.csv')


## 2. Головна порівняльна таблиця: всі моделі × всі метрики


In [ ]:
combined = results_a_h1.join(results_b_h1, lsuffix='_A', rsuffix='_B')
combined
combined.to_latex('results_table.tex')


## 3. h=1 vs h=7 comparison: grouped bar chart


In [ ]:
comparison = compare_horizons(results_a_h1, results_a_h7)
fig = px.bar(comparison.reset_index(), x='model', y=['MASE_h1', 'MASE_h7'], barmode='group')
fig.show()


## 4. Rolling vs Expanding window: comparison table


In [ ]:
rolling_path = processed_dir / 'results_experiment_a_h1.csv'
expanding_path = processed_dir / 'results_experiment_a_h1_expanding.csv'
if expanding_path.exists():
    rolling = pd.read_csv(rolling_path, index_col=0)
    expanding = pd.read_csv(expanding_path, index_col=0)
    window_comparison = compare_window_types(rolling, expanding)
    window_comparison
else:
    print('Expanding window results not found.')


## 5. Regime analysis table (post_covid / ukraine_war / normalization)


In [ ]:
regimes


## 6. Висновки: які моделі бють Random Walk, на яких режимах, при якому горизонті

- Перевірити `beats_rw` у Experiment A для кожного горизонту.
- Проаналізувати regime table для стабільності.
- Відзначити моделі зі стійким MASE < 1.0.


## 7. Feature importance evolution summary (з audit logs)


In [ ]:
import json
audit_files = sorted(processed_dir.glob('audit_*_h1.json'))
if not audit_files:
    print('No audit logs found.')
else:
    audit = json.loads(audit_files[0].read_text())
    counts = {}
    for entry in audit:
        for feat in entry['selected_features']:
            counts[feat] = counts.get(feat, 0) + 1
    summary = pd.DataFrame({'feature': list(counts.keys()), 'count': list(counts.values())})
    summary = summary.sort_values('count', ascending=False)
    summary.head(20)
